In [1]:
# %%
import argparse
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm
COUNT = "count [-]"
PROJ = "SegmentsProj"
PROJORTHO = PROJ + "Ortho"
PROJLOXO = PROJ + "Loxo"
PROJORTHONOTLOXO = PROJORTHO + "NotLoxo"
PROJLOXONOTORTHO = PROJLOXO + "NotOrtho"
BASELINE = "SegmentsBaseline"
CONFLICT = "SegmentsDeconfliction"


def read_detected(fname):
    df = pd.read_parquet(fname)
    if "date" not in df:
        df["date"] = df["start"].astype("datetime64[s]").dt.date
    df["length"] = df["stop"] - df["start"]
    df["length_min"] = df["length"] / 60

    df["datetime_start"] = df["start"].astype("datetime64[s]")  # .dt.date
    df["datetime_stop"] = df["stop"].astype("datetime64[s]")  # .dt.date
    return df.sort_values(["icao24", "start"])


def extract_loxo_not_ortho(df, args):
    return (
        df.query("iswhat=='loxodromy'")
        .query("dolmax>=@args.dolmax")
        .query("dlmax<@args.r*domax")
        .query("dlmax<@args.r*dolmax")
    )


def extract_ortho_not_loxo(df, args):
    return (
        df.query("iswhat=='orthodromy'")
        .query("dolmax>=@args.dolmax")
        .query("domax<@args.r*dlmax")
        .query("domax<@args.r*dolmax")
    )


def extract_ortho(df):
    return df.query(
        "iswhat=='orthodromy'"
    )  # .query("domax<100")#.query("domax<@args.r*dlmax")#.query("npts>10")#.query("dolmax>20")


def extract_loxo(df):
    return df.query(
        "iswhat=='loxodromy'"
    )  # .query("domax<100")#.query("domax<@args.r*dlmax")#.query("npts>10")#.query("dolmax>20")


def isole_altitude_dataset(df):
    return (
        df.query("altitude_start>=20000")
        .query("altitude_stop>=20000")
        .query("abs(altitude_stop-altitude_start)<200")
        .query("length>30")
    )


def intersection(l, q):
    start = max(q.start, l.start)
    end = min(l.stop, q.stop)
    return max(end - start, 0.0)


def union(l, q):
    start = min(q.start, l.start)
    end = max(l.stop, q.stop)
    return max(end - start, 0.0)


def is_included(l, q):
    return q.start <= l.start and l.stop <= q.stop


def inclusion_ratio(l, q):
    if is_included(l, q):
        return 1
    else:
        return intersection(l, q) / (l.stop - l.start)  # /union(l,q)


# def inclusion(l,q):# l C q ???
#     if is_included(l,q):
#         return 1
#     else:
#         inter=intersection(l,q)
#         if inter==0:
#             return 0
#         else:
#             return inter/(l.stop-l.start)


def getkey(line):
    return (line.icao24, line.start, line.stop)


def map_key(d, f):
    res = {}
    for k, v in d.items():
        res[f[k]] = v
    return res


def add_intersection(af, cf, suffix=""):
    res = {}
    d = {k: k + suffix for k in ["iou", "inclusion_ratio", "inclusion"]}
    af[d["iou"]] = 0.0
    af[d["inclusion_ratio"]] = 0.0
    # af[d["inclusion"]]=0.
    for _, line in tqdm.tqdm(cf.iterrows()):
        k = getkey(line)
        res[k] = []
        qf = af.query("date==@line.date").query("icao24==@line.icao24")
        for _, qline in qf.iterrows():
            leninter = intersection(qline, line)
            if leninter > 0.0:
                af.loc[qline.name, d["iou"]] = max(
                    leninter / union(qline, line), af.loc[qline.name, d["iou"]]
                )
                af.loc[qline.name, d["inclusion_ratio"]] = max(
                    inclusion_ratio(qline, line),
                    af.loc[qline.name, d["inclusion_ratio"]],
                )
                # af.loc[qline.name,d["inclusion"]]=max(inclusion(qline,line),af.loc[qline.name,d["inclusion"]])
                res[k].append(qline)
    return res


def plothist(d_ortho, vstr, ystr, bins=50, semilog=False):
    if semilog:
        bins = np.geomspace(
            min(v[vstr].min() for v in d_ortho.values()),
            max(v[vstr].max() for v in d_ortho.values()),
            bins + 1,
        )
    if isinstance(vstr, str):
        plt.hist(tuple(v[vstr] for k, v in d_ortho.items()), bins=bins)
    else:
        plt.hist(tuple(v[vstr[k]] for k, v in d_ortho.items()), bins=bins)
    if semilog:
        plt.xscale("log")
        ystr += " (log scale)"
    plt.xlabel(ystr)
    plt.ylabel(COUNT)
    plt.gca().legend(list(d_ortho.keys()))


def savefig(fig, fname, width=4):
    fig.set_tight_layout({"pad": 0})
    fig.set_figwidth(width)
    plt.savefig(f"{fname}", dpi=300, bbox_inches="tight")
    plt.clf()


def savenumber(s, fname):
    with open(fname + ".tex", "w") as f:
        f.write(s)

In [ ]:
from subprocess import call
for (threshiou,threshborder,threshslope) in itertools.product(lthreshiou,lthreshborder,lthreshslope):
    call(["make","-j4","second",f"THRESH_IOU={threshiou}", f"THRESH_BORDER={threshborder}" ,f"THRESH_SLOPE={threshslope}"])

In [218]:
dbaselineall = dict()
dotherall = dict()

In [397]:
from figures import read_config
import itertools
config = read_config()
print(config)
def filterd(d,tobe):
    return {k:v for k,v in d.items() if k in tobe}
lthreshiou=[0,0.05]#[0.05, 0.1, 0.2,0.25]0.0125,0.15,
lthreshborder=[0.1,0.15,0.2]#[0.05, 0.1, 0.2,0.25]
lthreshslope=[0.001,0.0005,0.00025]# ,0.0015]# 0.002,0.0025]
tobecomputed = set()
for (threshiou,threshborder,threshslope) in itertools.product(lthreshiou,lthreshborder,lthreshslope):
    key = (threshiou,threshborder,threshslope)
    tobecomputed.add(key)
    if key not in dbaselineall:
        assert (key not in dotherall)
        other = read_detected(f"{config.FOLDER}/trajs_detected_alpha_mean_slope_3600_0.01_0.5_{threshiou}_{threshslope}_{threshborder}")
        ref = read_detected(f"{config.FOLDER}/trajs_detectedref_alpha_mean_slope_3600_0.01_1_200")
        ref["thresh_iou"]=threshiou
        ref["thresh_border"]=threshborder
        ref["thresh_slope"]=threshslope    
        d = {k: isole_altitude_dataset(v) for k, v in {BASELINE: ref, PROJ: other}.items()}
        d[PROJORTHO] = extract_ortho(d[PROJ])
        add_intersection(d[BASELINE], d[PROJORTHO], suffix=PROJORTHO)
        add_intersection(d[PROJORTHO], d[BASELINE], suffix=BASELINE)
        dbaselineall[key]=d[BASELINE]
        dotherall[key]=d[PROJORTHO]


config(FOLDER='outfiles', FOLDER_DETECTEDREF_JSON='smallpartofthecataloguejusttoreproducethefigures/json/2207', FOLDER_FIGURES_PREFIX='figures')


6080it [01:08, 88.74it/s]
/tmp/ipykernel_3551013/2272018258.py:116: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  af[d["iou"]] = 0.0
/tmp/ipykernel_3551013/2272018258.py:117: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  af[d["inclusion_ratio"]] = 0.0
10191it [01:34, 107.61it/s]
11950it [02:17, 86.87it/s]
/tmp/ipykernel_3551013/2272018258.py:116: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveat

In [398]:
dbaseline=filterd(dbaselineall,tobecomputed)
dother=filterd(dotherall,tobecomputed)
base = pd.concat(dbaseline.values())
other = pd.concat(dother.values())
print(other.shape,base.shape)
#o=other.query("length>0").groupby(["thresh_iou","thresh_border","thresh_slope"]).iouSegmentsBaseline.mean()#.iouSegmentsProjOrtho.mean()
o=other.query("length>120").groupby(["thresh_iou","thresh_border","thresh_slope"]).iouSegmentsBaseline.mean()#.iouSegmentsProjOrtho.mean()
#o=other.query("length>0").query("iouSegmentsBaseline>0.8").groupby(["thresh_iou","thresh_border","thresh_slope"]).iouSegmentsBaseline.count()#.iouSegmentsProjOrtho.mean()
o

(162167, 36) (183438, 46)


thresh_iou  thresh_border  thresh_slope
0.00        0.10           0.00025         0.731077
                           0.00050         0.656621
                           0.00100         0.577904
            0.15           0.00025         0.713998
                           0.00050         0.644118
                           0.00100         0.571914
            0.20           0.00025         0.693618
                           0.00050         0.630664
                           0.00100         0.566092
0.05        0.10           0.00025         0.729253
                           0.00050         0.654278
                           0.00100         0.575395
            0.15           0.00025         0.711710
                           0.00050         0.640383
                           0.00100         0.567263
            0.20           0.00025         0.690690
                           0.00050         0.626105
                           0.00100         0.559204
Name: iouSegmentsBaselin

In [399]:
#b=base.query("length>0").groupby(["thresh_iou","thresh_border","thresh_slope"]).iouSegmentsProjOrtho.mean()#.iouSegmentsProjOrtho.mean()
b=base.query("length>120").groupby(["thresh_iou","thresh_border","thresh_slope"]).iouSegmentsProjOrtho.mean()#.iouSegmentsProjOrtho.mean()
#b=base.query("length>0").query("iouSegmentsProjOrtho>0.8").groupby(["thresh_iou","thresh_border","thresh_slope"]).iouSegmentsProjOrtho.count()#.iouSegmentsProjOrtho.mean()

b

thresh_iou  thresh_border  thresh_slope
0.00        0.10           0.00025         0.527163
                           0.00050         0.629179
                           0.00100         0.716726
            0.15           0.00025         0.542617
                           0.00050         0.644631
                           0.00100         0.730683
            0.20           0.00025         0.555696
                           0.00050         0.656547
                           0.00100         0.737868
0.05        0.10           0.00025         0.527369
                           0.00050         0.629322
                           0.00100         0.716868
            0.15           0.00025         0.542742
                           0.00050         0.644775
                           0.00100         0.730973
            0.20           0.00025         0.555808
                           0.00050         0.656681
                           0.00100         0.738166
Name: iouSegmentsProjOrt

In [403]:
df=np.minimum(o,b).rename("criteria").reset_index()#.query("thresh_slope<=0.002").query("thresh_iou<=0.2").query("thresh_border<=0.2")
#df=(o+b).rename("minIoU").reset_index()#.query("thresh_slope<=0.002").query("thresh_iou<=0.2").query("thresh_border<=0.2")
#np.minimum(o+b)#.idxmax()
print(df.criteria.idxmax())
df

4


,thresh_iou,thresh_border,thresh_slope,criteria
0,0.00,0.10,0.00025,0.527163
1,0.00,0.10,0.00050,0.629179
2,0.00,0.10,0.00100,0.577904
3,0.00,0.15,0.00025,0.542617
4,0.00,0.15,0.00050,0.644118
5,0.00,0.15,0.00100,0.571914
6,0.00,0.20,0.00025,0.555696
7,0.00,0.20,0.00050,0.630664
8,0.00,0.20,0.00100,0.566092
9,0.05,0.10,0.00025,0.527369


In [405]:
chart = alt.Chart(df).mark_point(strokeWidth=3).encode(
    x=alt.X("thresh_slope:Q",scale=alt.Scale(zero=False)),
    y=alt.Y("criteria:Q",scale=alt.Scale(zero=False)),
    color=alt.Color("thresh_iou:N", title="thresh_iou"),
    shape=alt.Shape("thresh_border:N", title="thresh_border"),
#    tooltip=[ "thresh_slope", "minIoU"]
)
chart.save(f"{config.FOLDER}/parameterstuning.pdf")
chart

alt.Chart(...)

In [152]:
(o+b).idxmax()

(np.float64(0.05), np.float64(0.1), np.float64(0.0005))

In [ ]:
print(len(dbaselineall))
del dbaselineall[(0.15,0.025,0.001)]
print(len(dbaselineall))

In [268]:
trajs=pd.read_parquet("/home/alligier/workInProgress/identifyingortholoxo/outfiles/trajs_man_3600")